In [1]:
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!pip install -q transformers datasets accelerate scikit-learn joblib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.2/253.2 MB 5.4 MB/s eta 0:00:00


In [1]:
# !pip install -q transformers datasets accelerate scikit-learn joblib pandas

import os
import joblib
import numpy as np
import pandas as pd
import torch

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [2]:
uploaded = files.upload()

csv_file = list(uploaded.keys())[0]
print("Uploaded file:", csv_file)

df = pd.read_csv(csv_file)

print("Columns in file:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())

Saving bert_3class_chunk_dataset.csv to bert_3class_chunk_dataset.csv
Uploaded file: bert_3class_chunk_dataset.csv
Columns in file:
Index(['file_name', 'class', 'chunk_id', 'chunk_text', 'chunk_chars'], dtype='object')

First 5 rows:
      file_name     class  chunk_id  \
0  Prot_009.pdf  protocol        54   
1  Prot_007.pdf  protocol        60   
2  Prot_001.pdf  protocol        18   
3  Prot_005.pdf  protocol        51   
4  Prot_007.pdf  protocol        84   

                                          chunk_text  chunk_chars  
0  pd - l2 ). based on preclinical in vitro data,...         2024  
1  regulatory requirements. clinical supplies sou...         1696  
2  figures figure 1 study scheme....................         1758  
3  are required by the study protocol, including ...         2173  
4  any aes that have an underlying true incidence...         2110  


In [3]:

possible_file_cols = ["file_name", "filename", "document_name", "doc_name", "source_file"]
possible_text_cols = ["chunk_text", "text", "content", "page_text"]
possible_label_cols = ["class", "label", "category", "tmf_class"]

file_col = None
text_col = None
label_col = None

for col in possible_file_cols:
    if col in df.columns:
        file_col = col
        break

for col in possible_text_cols:
    if col in df.columns:
        text_col = col
        break

for col in possible_label_cols:
    if col in df.columns:
        label_col = col
        break

if file_col is None:
    raise ValueError("No file name column found. Rename your document/file column to 'file_name'.")

if text_col is None:
    raise ValueError("No text column found. Rename your chunk text column to 'chunk_text'.")

if label_col is None:
    raise ValueError("No label column found. Rename your class column to 'class'.")

print("Using file column:", file_col)
print("Using text column:", text_col)
print("Using label column:", label_col)

# Standardize column names
df = df.rename(columns={
    file_col: "file_name",
    text_col: "chunk_text",
    label_col: "class"
})

df = df[["file_name", "chunk_text", "class"]].copy()

df = df.dropna(subset=["file_name", "chunk_text", "class"]).reset_index(drop=True)

df["file_name"] = df["file_name"].astype(str)
df["chunk_text"] = df["chunk_text"].astype(str)
df["class"] = df["class"].astype(str)

print("\nFinal columns:")
print(df.columns)

print("\nClass distribution by chunks:")
print(df["class"].value_counts())

print("\nNumber of unique documents:")
print(df["file_name"].nunique())


Using file column: file_name
Using text column: chunk_text
Using label column: class

Final columns:
Index(['file_name', 'chunk_text', 'class'], dtype='object')

Class distribution by chunks:
class
protocol                     700
safety_report                700
statistical_analysis_plan    700
Name: count, dtype: int64

Number of unique documents:
44


In [4]:
doc_df = df[["file_name", "class"]].drop_duplicates(subset=["file_name"])

print("\nClass distribution by documents:")
print(doc_df["class"].value_counts())

train_docs, test_docs = train_test_split(
    doc_df,
    test_size=0.2,
    random_state=42,
    stratify=doc_df["class"]
)

train_df = df[df["file_name"].isin(train_docs["file_name"])].reset_index(drop=True)
test_df = df[df["file_name"].isin(test_docs["file_name"])].reset_index(drop=True)

print("\nTrain documents:", train_df["file_name"].nunique())
print("Test documents:", test_df["file_name"].nunique())
print("Train chunks:", len(train_df))
print("Test chunks:", len(test_df))

overlap = set(train_df["file_name"]) & set(test_df["file_name"])
print("Document overlap between train and test:", len(overlap))



Class distribution by documents:
class
protocol                     15
safety_report                15
statistical_analysis_plan    14
Name: count, dtype: int64

Train documents: 35
Test documents: 9
Train chunks: 1656
Test chunks: 444
Document overlap between train and test: 0


In [5]:
label_encoder = LabelEncoder()

train_df["labels"] = label_encoder.fit_transform(train_df["class"])
test_df["labels"] = label_encoder.transform(test_df["class"])

num_labels = len(label_encoder.classes_)

label2id = {label: int(i) for i, label in enumerate(label_encoder.classes_)}
id2label = {int(i): label for i, label in enumerate(label_encoder.classes_)}

print("\nClasses:")
print(label_encoder.classes_)

print("\nlabel2id:")
print(label2id)


Classes:
['protocol' 'safety_report' 'statistical_analysis_plan']

label2id:
{'protocol': 0, 'safety_report': 1, 'statistical_analysis_plan': 2}


In [6]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df[["chunk_text", "labels"]])
test_dataset = Dataset.from_pandas(test_df[["chunk_text", "labels"]])

def tokenize_function(batch):
    return tokenizer(
        batch["chunk_text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["chunk_text"])
test_dataset = test_dataset.remove_columns(["chunk_text"])

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/1656 [00:00<?, ? examples/s]

Map:   0%|          | 0/444 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

training_args = TrainingArguments(
    output_dir="bioclinicalbert_tmf_classifier_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.547805,0.553643,0.720721,0.693530
2,0.345865,0.562940,0.707207,0.663908
3,0.218232,0.549500,0.725225,0.683018


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=621, training_loss=0.4641117366231774, metrics={'train_runtime': 702.4072, 'train_samples_per_second': 7.073, 'train_steps_per_second': 0.884, 'total_flos': 1307147459272704.0, 'train_loss': 0.4641117366231774, 'epoch': 3.0})

In [10]:
results = trainer.evaluate()

print("\nEvaluation Results:")
print(results)

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\nAccuracy:")
print(accuracy_score(y_true, y_pred))

print("\nMacro F1:")
print(f1_score(y_true, y_pred, average="macro"))

print("\nPer-class Precision / Recall / F1:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)

# Optional readable confusion matrix
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{c}" for c in label_encoder.classes_],
    columns=[f"Predicted_{c}" for c in label_encoder.classes_]
)

print("\nReadable Confusion Matrix:")
print(cm_df)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.218232,0.553643,3,0.720721,0.693530



Evaluation Results:
{'eval_loss': 0.5536425709724426, 'eval_accuracy': 0.7207207207207207, 'eval_macro_f1': 0.6935300456244126}



Accuracy:
0.7207207207207207

Macro F1:
0.6935300456244126

Per-class Precision / Recall / F1:
                           precision    recall  f1-score   support

                 protocol       0.69      0.38      0.49       140
            safety_report       0.99      0.91      0.95       174
statistical_analysis_plan       0.52      0.83      0.64       130

                 accuracy                           0.72       444
                macro avg       0.73      0.71      0.69       444
             weighted avg       0.76      0.72      0.71       444


Confusion Matrix:
[[ 53   0  87]
 [  4 159  11]
 [ 20   2 108]]

Readable Confusion Matrix:
                                  Predicted_protocol  Predicted_safety_report  \
Actual_protocol                                   53                        0   
Actual_safety_report                               4                      159   
Actual_statistical_analysis_plan                  20                        2   

              

In [11]:
# ==========================================
# Document-level majority voting evaluation
# ==========================================

test_df_eval = test_df.copy()

test_df_eval["true_label"] = label_encoder.inverse_transform(y_true)
test_df_eval["pred_label"] = label_encoder.inverse_transform(y_pred)

doc_level_preds = (
    test_df_eval
    .groupby("file_name")
    .agg(
        true_label=("true_label", "first"),
        pred_label=("pred_label", lambda x: x.value_counts().idxmax()),
        total_chunks=("pred_label", "count")
    )
    .reset_index()
)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("Document-level Accuracy:")
print(accuracy_score(doc_level_preds["true_label"], doc_level_preds["pred_label"]))

print("\nDocument-level Macro F1:")
print(f1_score(doc_level_preds["true_label"], doc_level_preds["pred_label"], average="macro"))

print("\nDocument-level Classification Report:")
print(classification_report(doc_level_preds["true_label"], doc_level_preds["pred_label"]))

print("\nDocument-level Confusion Matrix:")
print(confusion_matrix(doc_level_preds["true_label"], doc_level_preds["pred_label"]))

doc_level_preds.head(20)

Document-level Accuracy:
0.7777777777777778

Document-level Macro F1:
0.75

Document-level Classification Report:
                           precision    recall  f1-score   support

                 protocol       1.00      0.33      0.50         3
            safety_report       1.00      1.00      1.00         3
statistical_analysis_plan       0.60      1.00      0.75         3

                 accuracy                           0.78         9
                macro avg       0.87      0.78      0.75         9
             weighted avg       0.87      0.78      0.75         9


Document-level Confusion Matrix:
[[1 0 2]
 [0 3 0]
 [0 0 3]]


,file_name,true_label,pred_label,total_chunks
0,Prot_005.pdf,protocol,statistical_analysis_plan,37
1,Prot_011.pdf,protocol,protocol,29
2,Prot_013.pdf,protocol,statistical_analysis_plan,74
3,sap-003.pdf,statistical_analysis_plan,statistical_analysis_plan,59
4,sap-008.pdf,statistical_analysis_plan,statistical_analysis_plan,14
5,sap_009.pdf,statistical_analysis_plan,statistical_analysis_plan,57
6,sr-003.pdf,safety_report,safety_report,95
7,sr-012.pdf,safety_report,safety_report,47
8,sr-014.pdf,safety_report,safety_report,32


In [12]:
save_dir = "saved_bioclinicalbert_tmf_3class"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

joblib.dump(label_encoder, "label_encoder.pkl")

print("\nSaved model folder:", save_dir)
print("Saved label encoder: label_encoder.pkl")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved model folder: saved_bioclinicalbert_tmf_3class
Saved label encoder: label_encoder.pkl


In [13]:
!zip -r saved_bioclinicalbert_tmf_3class.zip saved_bioclinicalbert_tmf_3class label_encoder.pkl

files.download("saved_bioclinicalbert_tmf_3class.zip")

  adding: saved_bioclinicalbert_tmf_3class/ (stored 0%)
  adding: saved_bioclinicalbert_tmf_3class/training_args.bin (deflated 51%)
  adding: saved_bioclinicalbert_tmf_3class/tokenizer_config.json (deflated 43%)
  adding: saved_bioclinicalbert_tmf_3class/tokenizer.json (deflated 70%)
  adding: saved_bioclinicalbert_tmf_3class/model.safetensors (deflated 7%)
  adding: saved_bioclinicalbert_tmf_3class/config.json (deflated 54%)
  adding: label_encoder.pkl (deflated 35%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>